<a href="https://colab.research.google.com/github/Angela-K-C/SemProject_MLEngine/blob/main/LightGBM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
%pip install -q lightgbm scikit-learn matplotlib seaborn

In [3]:
from pathlib import Path

RAW_DATA_DIR = Path("/content/drive/MyDrive/M5_Dataset")

SALES_FILE = RAW_DATA_DIR / "sales_train_evaluation.csv"
CALENDAR_FILE = RAW_DATA_DIR / "calendar.csv"
PRICES_FILE = RAW_DATA_DIR / "sell_prices.csv"

In [4]:
# Imports and configs
import gc
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import root_mean_squared_error, mean_absolute_error

In [5]:
# setting variables for my series and when to make the split

MAX_SERIES = 500
VALIDATION_START, VALIDATION_END = 1914, 1941

Series here don't represent rows. The columns were melted. It will be at least 500,000 rows.

In [6]:
calendar = pd.read_csv(CALENDAR_FILE)
sales = pd.read_csv(SALES_FILE)

In [7]:
if MAX_SERIES and MAX_SERIES < len(sales):
    sales = sales.sample(
        n=MAX_SERIES,
        random_state=42
    ).reset_index(drop=True)

Selecting the number of series we'll be using during training. Random_state=42 means that the same data can be reselected even without changing the input

In [8]:
# picking out the columns with days
day_cols = [
    col for col in sales.columns
    if col.startswith("d_")
]

In [9]:
# Melting columns
df = pd.melt(
    sales,
    id_vars=[
        "id", "item_id", "dept_id",
        "cat_id", "store_id", "state_id"
    ],
    value_vars=day_cols,
    var_name="d",
    value_name="sales"
)

Transforms the multiple columns into rows (Melting).

In [10]:
del sales
gc.collect()

90

sales variable is deleted because df contains the same thing. gc then collects objeccts that are not needed, helps with memory pressure.

In [11]:
df = df.merge(
    calendar,
    on="d",
    how="left"
)

Left join merge with the calendar for dates.

In [12]:
prices = pd.read_csv(PRICES_FILE)

prices = prices[
    prices["item_id"].isin(df["item_id"].unique())
    & prices["store_id"].isin(df["store_id"].unique())
]

Filtering Prices to make sure that we don't get unecessary data since we were using a limited series (300) in this case.

In [13]:
df = df.merge(
    prices,
    on=["store_id", "item_id", "wm_yr_wk"],
    how="left"
)

In [14]:
del calendar, prices
gc.collect()

0

2nd garbage collection

In [15]:
df["day_num"] = (
    df["d"].str.extract(r"(\d+)")[0].astype(np.int32)
)

Extracting days because they look like d_1 and the desired output is 1. Also saved on memory by converting them to int.32

In [16]:
df = df.sort_values(
    ["id", "day_num"]
).reset_index(drop=True)

Ordering items by ID then the day number

In [17]:
cat_cols = [
    "id", "item_id", "dept_id", "cat_id",
    "store_id", "state_id", "weekday",
    "event_name_1", "event_type_1",
    "event_name_2", "event_type_2"
]

for col in cat_cols:
    if col in df.columns:
        df[col] = df[col].astype("category")

Making these columns into categories since they have no continuous numerical measurement. Makes our algorithm process them slightly faster.

In [18]:
df["sales_lag_28"] = (
    df.groupby("id", observed=True)["sales"].shift(28)
)

Creates a new column called sales lag that has data from 28 days ago

In [19]:
df["rolling_mean_7"] = (
    df.groupby("id", observed=True)["sales"].transform(
        lambda s: s.shift(28).rolling(7, min_periods=7).mean()
    )
)

This is averaging sales for 7 days

In [20]:
train_df = df[
    df["day_num"] < VALIDATION_START
].dropna(subset=["sales_lag_28", "rolling_mean_7"])

Splits data, we earlier defined the window. Removes the data without observations for sales lag and rolling mean

In [21]:
val_df = df[
    df["day_num"].between(
        VALIDATION_START, VALIDATION_END
    )
].dropna(subset=["sales_lag_28", "rolling_mean_7"])

validation data set, dropping rows without sales lag and 7 day mean too

In [22]:
feature_cols = [
    c for c in df.columns
    if c not in {"sales", "d", "day_num", "date"}
]

In [23]:
X_train, y_train = (
    train_df[feature_cols], train_df["sales"]
)

X_val, y_val = (
    val_df[feature_cols], val_df["sales"]
)

X the input info the model uses to make predictions,
Y the correct answer the model should learn to predict

In [24]:
model = lgb.LGBMRegressor(
    objective="regression",
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=31,
    n_jobs=-1,
    random_state=42,
    verbosity=-1
)

regression - for continuous data.

leaves are limited to 31, it is a common default for lightGBM

In [25]:

model.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    eval_metric="rmse",
    callbacks=[
        lgb.early_stopping(30),
        lgb.log_evaluation(25)
    ]
)

Training until validation scores don't improve for 30 rounds
[25]	valid_0's rmse: 1.95273	valid_0's l2: 3.81315
[50]	valid_0's rmse: 1.83545	valid_0's l2: 3.36888
[75]	valid_0's rmse: 1.83797	valid_0's l2: 3.37813
Early stopping, best iteration is:
[58]	valid_0's rmse: 1.83222	valid_0's l2: 3.35703


LGBMRegressor(learning_rate=0.05, n_estimators=1000, n_jobs=-1,
              objective='regression', random_state=42, verbosity=-1)

In [26]:
preds = model.predict(
    X_val,
    num_iteration=model.best_iteration_
)

In [39]:
comparison_df = pd.DataFrame({
    "Actual Sales": y_val.to_numpy(),
    "Predicted Sales": preds
})

display(comparison_df.round(3))

,Actual Sales,Predicted Sales
0,0,0.404
1,0,0.404
2,0,0.338
3,0,0.338
4,0,0.338
...,...,...
13995,0,0.219
13996,0,0.241
13997,0,0.238
13998,0,0.251


In [27]:
baseline_preds = X_val["sales_lag_28"].to_numpy()

In [38]:
print(
    f"LightGBM RMSE: "
    f"{root_mean_squared_error(y_val, preds):.4f}"
)
print(
    f"LightGBM MAE: "
    f"{mean_absolute_error(y_val, preds):.4f}"
)

LightGBM RMSE: 1.8322
LightGBM MAE: 0.9734


In [35]:

baseline_preds = X_val["sales_lag_28"].to_numpy()

baseline_rmse = root_mean_squared_error(y_val, baseline_preds)
baseline_mae = mean_absolute_error(y_val, baseline_preds)

print("Lag-28 Baseline RMSE:", round(baseline_rmse, 4))
print("Lag-28 Baseline MAE:", round(baseline_mae, 4))


Lag-28 Baseline RMSE: 2.3626
Lag-28 Baseline MAE: 1.1541


In [37]:

from sklearn.linear_model import Ridge
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import root_mean_squared_error, mean_absolute_error
import pandas as pd


categorical_cols = X_train.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

numerical_cols = X_train.select_dtypes(
    include=["number"]
).columns.tolist()


numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])


categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(
        strategy="constant", fill_value="MISSING"
    )),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        max_categories=100
    ))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numerical_cols),
    ("categorical", categorical_pipeline, categorical_cols)
])


ridge_model = Pipeline([
    ("preprocessor", preprocessor),
    ("ridge", Ridge(alpha=1.0))
])


ridge_model.fit(X_train, y_train)


ridge_preds = ridge_model.predict(X_val)


ridge_rmse = root_mean_squared_error(y_val, ridge_preds)
ridge_mae = mean_absolute_error(y_val, ridge_preds)

print("Ridge Regression RMSE:", round(ridge_rmse, 4))
print("Ridge Regression MAE:", round(ridge_mae, 4))


Ridge Regression RMSE: 1.9241
Ridge Regression MAE: 1.0068


In [34]:

from sklearn.ensemble import RandomForestRegressor
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import root_mean_squared_error, mean_absolute_error


categorical_cols = X_train.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

numerical_cols = X_train.select_dtypes(
    include=["number"]
).columns.tolist()


numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])



categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant",
                              fill_value="MISSING")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numerical_cols),
    ("categorical", categorical_pipeline, categorical_cols)
])

model = Pipeline([
    ("preprocessor", preprocessor),
    ("random_forest", RandomForestRegressor(
        n_estimators=30,
        max_depth=12,
        min_samples_leaf=10,
        max_features=0.7,
        n_jobs=-1,
        random_state=42
    ))
])

model.fit(X_train, y_train)

preds = model.predict(X_val)

print("Random Forest RMSE:",
      root_mean_squared_error(y_val, preds))
print("Random Forest MAE:",
      mean_absolute_error(y_val, preds))


KeyboardInterrupt: 